<a href="https://colab.research.google.com/github/jaimeisaac2020/Python-analsisis-basicos/blob/mi-github/pronosticos_tesis_amazon_chronos_libreria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Paso 1: Instalar las librerías necesarias
# Descomenta las siguientes líneas si no las has instalado
!pip install torch
!pip install chronos-forecasting
!pip install accelerate

In [ ]:
import pandas as pd
import torch
from chronos import BaseChronosPipeline
import matplotlib.pyplot as plt
import warnings

# Ignorar advertencias para una salida más limpia
warnings.filterwarnings("ignore")

In [ ]:
try:
    # --- 1. Cargar y Preparar los Datos ---
    print("Cargando el dataset 'dataset_completo_google_amazon.csv'...")
    df = pd.read_csv('dataset_completo_google_amazon.csv')

    # Convertir la columna 'Date' a formato datetime y establecerla como índice
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)

    # Seleccionar la serie de tiempo del precio de cierre de Google
    google_series = df['GOOGL_Close']

    # Asegurarnos de que la serie tenga una frecuencia de días hábiles ('B')
    # y rellenar cualquier posible hueco (aunque tu dataset parece no tenerlos)
    google_series = google_series.asfreq('B').ffill()

    print(f"Datos históricos cargados. La serie contiene {len(google_series)} puntos de datos de días hábiles.")
    print(f"El rango de fechas va de {google_series.index.min().date()} a {google_series.index.max().date()}.")
except FileNotFoundError:
    print("Error: El archivo 'dataset_completo_google_amazon.csv' no fue encontrado.")
except Exception as e:
    print(f"Ocurrió un error al cargar o preparar los datos: {e}")

In [ ]:
google_series.head()

In [ ]:
# --- 2. Configurar el Pipeline de Chronos ---
print("\nInicializando el pipeline de Amazon Chronos...")
print("Este paso puede tardar un poco mientras se descarga el modelo.")

# Nota: Si no tienes una GPU disponible, cambia "cuda" a "cpu".
# El proceso será más lento en CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.bfloat16,
)
print(f"Pipeline cargado y funcionando en '{device}'.")

In [ ]:
# --- 3. Generar Pronóstico para 2025 ---
# El contexto es toda la serie histórica disponible (2021-2024)
contexto = torch.tensor(google_series.values, dtype=torch.bfloat16)

# Calcular cuántos días hábiles hay en 2025 para la predicción
dias_habiles_2025 = pd.bdate_range(start='2025-01-01', end='2025-12-31')
prediction_length = len(dias_habiles_2025)

print(f"\nRealizando pronóstico para los {prediction_length} días hábiles de 2025...")

In [ ]:
# Predecir los cuantiles para obtener la mediana y el intervalo de confianza
quantiles_tensor, _ = pipeline.predict_quantiles(
    context=contexto.to(device),
    prediction_length=prediction_length,
    quantile_levels=[0.1, 0.5, 0.9],  # Cuantiles 10%, 50% (mediana) y 90%
)

# Mover los resultados de vuelta a la CPU y a formato NumPy para su manipulación
quantiles = quantiles_tensor.cpu().numpy()

print("Pronóstico completado.")

In [ ]:
    # --- 4. Procesar y Mostrar los Resultados ---
    # Extraer los cuantiles para el gráfico
    low, median, high = quantiles[0, :, 0], quantiles[0, :, 1], quantiles[0, :, 2]

    # Crear un DataFrame con el pronóstico para una visualización clara
    forecast_df = pd.DataFrame(
        data={'Baja (10%)': low, 'Mediana (50%)': median, 'Alta (90%)': high},
        index=dias_habiles_2025
    )

    print("\n--- Primeros 10 días del pronóstico para GOOGL en 2025 ---")
    print(forecast_df.head(10).to_string(float_format="%.2f"))
    print("\n--- Últimos 10 días del pronóstico para GOOGL en 2025 ---")
    print(forecast_df.tail(10).to_string(float_format="%.2f"))

In [ ]:
# --- 5. Visualizar los Resultados ---
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(15, 8))

# Graficar la serie histórica
ax.plot(google_series.index, google_series.values, color="royalblue", label="Precio Histórico")

# Graficar la predicción de la mediana
ax.plot(forecast_df.index, forecast_df['Mediana (50%)'], color="tomato", label="Pronóstico (Mediana)")

# Rellenar el área del intervalo de confianza
ax.fill_between(forecast_df.index,
                forecast_df['Baja (10%)'],
                forecast_df['Alta (90%)'],
                color="tomato", alpha=0.3, label="Intervalo de Predicción (80%)")

ax.set_title('Pronóstico de Precio de Cierre de GOOGL para 2025 (Frecuencia Diaria)', fontsize=16)
ax.set_xlabel("Fecha", fontsize=12)
ax.set_ylabel("Precio de Cierre (USD)", fontsize=12)
ax.legend()
ax.grid(True)

# Ajustar el límite del eje X para enfocar en el final del histórico y el pronóstico
ax.set_xlim([pd.to_datetime('2024-01-01'), pd.to_datetime('2025-12-31')])

plt.tight_layout()
plt.show()

In [ ]:
# Calcular el Error Cuadrático Medio (MSE)
# This code requires 'validation_data' and 'median_forecast' which are not defined in the current notebook.
# To calculate MSE, you would need a dataset with actual values for the forecast period to compare against the predicted values.
# mse = mean_squared_error(validation_data.values, median_forecast)

# print("\n--- Resultados de la Validación ---")
# print(f"Error Cuadrático Medio (MSE) del pronóstico: {mse:.4f}")

In [ ]:
# Paso 1: Instalar las librerías necesarias
# Si no las has instalado, descomenta y ejecuta estas líneas en tu entorno
# !pip install torch
# !pip install chronos-forecasting
# !pip install accelerate
# !pip install scikit-learn

import pandas as pd
import torch
from chronos import BaseChronosPipeline
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error
import warnings

# --- Configuración ---
warnings.filterwarnings("ignore")

try:
    # --- 1. Cargar y Preparar los Datos ---
    print("Cargando el dataset 'dataset_completo_google_amazon.csv'...")
    df = pd.read_csv('dataset_completo_google_amazon.csv')

    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)

    google_series = df['GOOGL_Close'].asfreq('B').ffill()

    print(f"Datos históricos cargados. La serie va de {google_series.index.min().date()} a {google_series.index.max().date()}.")

    # --- 2. Configurar el Pipeline de Chronos ---
    print("\nInicializando el pipeline de Amazon Chronos...")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    pipeline = BaseChronosPipeline.from_pretrained(
        "amazon/chronos-t5-small",
        device_map=device,
        torch_dtype=torch.bfloat16,
    )
    print(f"Pipeline cargado y funcionando en '{device}'.")

    # --- 3. Validación del Modelo y Cálculo del MSE ---
    print("\n--- Iniciando la fase de validación para calcular el error ---")

    # Definir el número de días para el conjunto de validación (aprox. 3 meses)
    validation_days = 66  # 22 días hábiles/mes * 3 meses

    # Dividir los datos
    train_data = google_series[:-validation_days]
    validation_data = google_series[-validation_days:]

    print(f"Usando datos hasta {train_data.index.max().date()} para entrenar.")
    print(f"Validando el pronóstico para el período de {validation_data.index.min().date()} a {validation_data.index.max().date()}.")

    # Preparar el contexto para el modelo
    validation_context = torch.tensor(train_data.values, dtype=torch.bfloat16)

    # Generar el pronóstico para el período de validación
    quantiles_tensor, _ = pipeline.predict_quantiles(
        context=validation_context.to(device),
        prediction_length=validation_days,
        quantile_levels=[0.5], # Solo necesitamos la mediana (mejor estimador puntual)
    )

    # Extraer la mediana de las predicciones
    median_forecast = quantiles_tensor.cpu().numpy()[0, :, 0]

    # Calcular el Error Cuadrático Medio (MSE)
    mse = mean_squared_error(validation_data.values, median_forecast)

    print("\n--- Resultados de la Validación ---")
    print(f"Error Cuadrático Medio (MSE) del pronóstico: {mse:.4f}")

    # --- 4. Generar Pronóstico Final para 2025 ---
    print("\n--- Iniciando el pronóstico final para 2025 (usando todos los datos históricos) ---")

    # El contexto es la serie histórica completa
    full_context = torch.tensor(google_series.values, dtype=torch.bfloat16)

    dias_habiles_2025 = pd.bdate_range(start='2025-01-01', end='2025-12-31')
    prediction_length = len(dias_habiles_2025)

    print(f"Realizando pronóstico para los {prediction_length} días hábiles de 2025...")

    # Predecir los cuantiles para 2025
    quantiles_tensor_2025, _ = pipeline.predict_quantiles(
        context=full_context.to(device),
        prediction_length=prediction_length,
        quantile_levels=[0.1, 0.5, 0.9],
    )

    quantiles_2025 = quantiles_tensor_2025.cpu().numpy()
    print("Pronóstico completado.")

    # --- 5. Procesar y Mostrar los Resultados de 2025 ---
    low, median, high = quantiles_2025[0, :, 0], quantiles_2025[0, :, 1], quantiles_2025[0, :, 2]

    forecast_df = pd.DataFrame(
        data={'Baja (10%)': low, 'Mediana (50%)': median, 'Alta (90%)': high},
        index=dias_habiles_2025
    )

    print("\n--- Primeros 10 días del pronóstico para GOOGL en 2025 ---")
    print(forecast_df.head(10).to_string(float_format="%.2f"))

    # --- 6. Visualizar el Pronóstico Final ---
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(15, 8))

    ax.plot(google_series.index, google_series.values, color="royalblue", label="Precio Histórico")
    ax.plot(forecast_df.index, forecast_df['Mediana (50%)'], color="tomato", label="Pronóstico 2025 (Mediana)")
    ax.fill_between(forecast_df.index,
                    forecast_df['Baja (10%)'],
                    forecast_df['Alta (90%)'],
                    color="tomato", alpha=0.3, label="Intervalo de Predicción (80%)")

    ax.set_title('Pronóstico de Precio de Cierre de GOOGL para 2025 con MSE de Validación', fontsize=16)
    ax.set_xlabel("Fecha", fontsize=12)
    ax.set_ylabel("Precio de Cierre (USD)", fontsize=12)
    ax.legend()
    ax.grid(True)

    ax.set_xlim([pd.to_datetime('2024-01-01'), pd.to_datetime('2025-12-31')])

    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"\nHa ocurrido un error inesperado: {e}")

In [ ]:
# Paso 1: Instalar las librerías necesarias
# Si no las has instalado, descomenta y ejecuta estas líneas en tu entorno
# !pip install torch
# !pip install chronos-forecasting
# !pip install accelerate
# !pip install scikit-learn

import pandas as pd
import torch
from chronos import BaseChronosPipeline
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error
import warnings

# --- Configuración ---
# Ignorar advertencias comunes para una salida más limpia
warnings.filterwarnings("ignore")

try:
    # --- 1. Cargar y Preparar los Datos ---
    print("Cargando el dataset 'dataset_completo_google_amazon.csv'...")
    df = pd.read_csv('dataset_completo_google_amazon.csv')

    # Convertir la columna 'Date' a formato datetime y establecerla como índice
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)

    # <<-- MODIFICACIÓN AQUÍ: Seleccionar la serie de tiempo de Amazon
    amazon_series = df['AMZN_Close'].asfreq('B').ffill()

    print(f"Datos históricos de Amazon cargados. La serie contiene {len(amazon_series)} puntos de datos de días hábiles.")
    print(f"El rango de fechas va de {amazon_series.index.min().date()} a {amazon_series.index.max().date()}.")

    # --- 2. Configurar el Pipeline de Chronos ---
    print("\nInicializando el pipeline de Amazon Chronos...")
    print("Este paso puede tardar un poco mientras se descarga el modelo.")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    pipeline = BaseChronosPipeline.from_pretrained(
        "amazon/chronos-t5-small",
        device_map=device,
        torch_dtype=torch.bfloat16,
    )
    print(f"Pipeline cargado y funcionando en '{device}'.")

    # --- 3. Validación del Modelo y Cálculo del MSE ---
    print("\n--- Iniciando la fase de validación para calcular el error ---")

    validation_days = 66  # Aprox. 3 meses de días hábiles

    # Dividir los datos de Amazon
    train_data = amazon_series[:-validation_days]
    validation_data = amazon_series[-validation_days:]

    print(f"Usando datos hasta {train_data.index.max().date()} para entrenar.")
    print(f"Validando el pronóstico para el período de {validation_data.index.min().date()} a {validation_data.index.max().date()}.")

    # Preparar el contexto para el modelo
    validation_context = torch.tensor(train_data.values, dtype=torch.bfloat16)

    # Generar el pronóstico para el período de validación
    quantiles_tensor, _ = pipeline.predict_quantiles(
        context=validation_context.to(device),
        prediction_length=validation_days,
        quantile_levels=[0.5], # Solo necesitamos la mediana
    )

    median_forecast = quantiles_tensor.cpu().numpy()[0, :, 0]

    # Calcular el Error Cuadrático Medio (MSE)
    mse = mean_squared_error(validation_data.values, median_forecast)

    print("\n--- Resultados de la Validación ---")
    print(f"Error Cuadrático Medio (MSE) del pronóstico de Amazon: {mse:.4f}")

    # --- 4. Generar Pronóstico Final para 2025 ---
    print("\n--- Iniciando el pronóstico final para 2025 (usando todos los datos históricos) ---")

    # El contexto es la serie histórica completa de Amazon
    full_context = torch.tensor(amazon_series.values, dtype=torch.bfloat16)

    dias_habiles_2025 = pd.bdate_range(start='2025-01-01', end='2025-12-31')
    prediction_length = len(dias_habiles_2025)

    print(f"Realizando pronóstico para los {prediction_length} días hábiles de 2025...")

    # Predecir los cuantiles para 2025
    quantiles_tensor_2025, _ = pipeline.predict_quantiles(
        context=full_context.to(device),
        prediction_length=prediction_length,
        quantile_levels=[0.1, 0.5, 0.9],
    )

    quantiles_2025 = quantiles_tensor_2025.cpu().numpy()
    print("Pronóstico completado.")

    # --- 5. Procesar y Mostrar los Resultados de 2025 ---
    low, median, high = quantiles_2025[0, :, 0], quantiles_2025[0, :, 1], quantiles_2025[0, :, 2]

    forecast_df = pd.DataFrame(
        data={'Baja (10%)': low, 'Mediana (50%)': median, 'Alta (90%)': high},
        index=dias_habiles_2025
    )

    print("\n--- Primeros 10 días del pronóstico para AMZN en 2025 ---")
    print(forecast_df.head(10).to_string(float_format="%.2f"))
    print("\n--- Últimos 10 días del pronóstico para AMZN en 2025 ---")
    print(forecast_df.tail(10).to_string(float_format="%.2f"))

    # --- 6. Visualizar el Pronóstico Final ---
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(15, 8))

    ax.plot(amazon_series.index, amazon_series.values, color="royalblue", label="Precio Histórico de Amazon")
    ax.plot(forecast_df.index, forecast_df['Mediana (50%)'], color="orange", label="Pronóstico para Amazon 2025 (Mediana)")
    ax.fill_between(forecast_df.index,
                    forecast_df['Baja (10%)'],
                    forecast_df['Alta (90%)'],
                    color="orange", alpha=0.3, label="Intervalo de Predicción (80%)")

    ax.set_title('Pronóstico de Precio de Cierre de AMZN para 2025 con MSE de Validación', fontsize=16)
    ax.set_xlabel("Fecha", fontsize=12)
    ax.set_ylabel("Precio de Cierre (USD)", fontsize=12)
    ax.legend()
    ax.grid(True)

    ax.set_xlim([pd.to_datetime('2024-01-01'), pd.to_datetime('2025-12-31')])

    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"\nHa ocurrido un error inesperado: {e}")
    print("Asegúrate de tener las librerías 'torch', 'chronos-forecasting' y 'accelerate' instaladas.")
    print("Si el error es de memoria (Out of Memory), puede que tu GPU o RAM no sea suficiente para esta operación.")

In [ ]:
# Paso 1: Instalar las librerías necesarias
# Si no las has instalado, descomenta y ejecuta estas líneas en tu entorno.
# !pip install torch
# !pip install chronos-forecasting
# !pip install accelerate
# !pip install scikit-learn

import pandas as pd
import torch
from chronos import BaseChronosPipeline
import matplotlib.pyplot as plt
import warnings

# --- Configuración ---
warnings.filterwarnings("ignore")

try:
    # --- 1. Cargar y Preparar los Datos ---
    print("Cargando el dataset 'dataset_completo_google_amazon.csv'...")
    df = pd.read_csv('dataset_completo_google_amazon.csv')

    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)

    # Preparar series para GOOGL y AMZN con frecuencia de días hábiles
    google_series = df['GOOGL_Close'].asfreq('B').ffill()
    amazon_series = df['AMZN_Close'].asfreq('B').ffill()

    print("Datos históricos para Google y Amazon cargados y preparados.")

    # --- 2. Configurar el Pipeline de Chronos ---
    print("\nInicializando el pipeline de Amazon Chronos (esto puede tardar)...")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    pipeline = BaseChronosPipeline.from_pretrained(
        "amazon/chronos-t5-small",
        device_map=device,
        torch_dtype=torch.bfloat16,
    )
    print(f"Pipeline cargado y funcionando en '{device}'.")

    # --- 3. Generar Pronóstico para 2025 para GOOGLE ---
    print("\nGenerando pronóstico para Google...")
    context_googl = torch.tensor(google_series.values, dtype=torch.bfloat16)

    dias_habiles_2025 = pd.bdate_range(start='2025-01-01', end='2025-12-31')
    prediction_length = len(dias_habiles_2025)

    quantiles_googl_tensor, _ = pipeline.predict_quantiles(
        context=context_googl.to(device),
        prediction_length=prediction_length,
        quantile_levels=[0.1, 0.5, 0.9],
    )
    quantiles_googl = quantiles_googl_tensor.cpu().numpy()

    low_g, median_g, high_g = quantiles_googl[0, :, 0], quantiles_googl[0, :, 1], quantiles_googl[0, :, 2]
    forecast_df_googl = pd.DataFrame(
        data={'Baja': low_g, 'Mediana': median_g, 'Alta': high_g},
        index=dias_habiles_2025
    )
    print("Pronóstico para Google completado.")

    # --- 4. Generar Pronóstico para 2025 para AMAZON ---
    print("Generando pronóstico para Amazon...")
    context_amzn = torch.tensor(amazon_series.values, dtype=torch.bfloat16)

    quantiles_amzn_tensor, _ = pipeline.predict_quantiles(
        context=context_amzn.to(device),
        prediction_length=prediction_length,
        quantile_levels=[0.1, 0.5, 0.9],
    )
    quantiles_amzn = quantiles_amzn_tensor.cpu().numpy()

    low_a, median_a, high_a = quantiles_amzn[0, :, 0], quantiles_amzn[0, :, 1], quantiles_amzn[0, :, 2]
    forecast_df_amzn = pd.DataFrame(
        data={'Baja': low_a, 'Mediana': median_a, 'Alta': high_a},
        index=dias_habiles_2025
    )
    print("Pronóstico para Amazon completado.")

    # --- 5. Visualizar los Resultados Combinados ---
    print("\nGenerando gráfico combinado...")
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(18, 9))

    # Graficar datos históricos
    ax.plot(google_series.index, google_series.values, color="blue", label="Google Histórico")
    ax.plot(amazon_series.index, amazon_series.values, color="orange", label="Amazon Histórico")

    # Graficar pronósticos (mediana)
    ax.plot(forecast_df_googl.index, forecast_df_googl['Mediana'], color="cyan", linestyle='--', label="Google Pronóstico 2025")
    ax.plot(forecast_df_amzn.index, forecast_df_amzn['Mediana'], color="red", linestyle='--', label="Amazon Pronóstico 2025")

    # Rellenar intervalos de confianza
    ax.fill_between(forecast_df_googl.index,
                    forecast_df_googl['Baja'],
                    forecast_df_googl['Alta'],
                    color="cyan", alpha=0.2, label="Google Int. de Predicción (80%)")

    ax.fill_between(forecast_df_amzn.index,
                    forecast_df_amzn['Baja'],
                    forecast_df_amzn['Alta'],
                    color="red", alpha=0.2, label="Amazon Int. de Predicción (80%)")

    # Configuración del gráfico
    ax.set_title('Histórico 2024 vs. Pronóstico 2025 para Google y Amazon', fontsize=18)
    ax.set_xlabel("Fecha", fontsize=14)
    ax.set_ylabel("Precio de Cierre (USD)", fontsize=14)

    # Establecer los límites del eje X desde 2024 hasta 2025
    ax.set_xlim([pd.to_datetime('2024-01-01'), pd.to_datetime('2025-12-31')])

    # Organizar la leyenda para que no se superponga
    handles, labels = ax.get_legend_handles_labels()
    order = [0, 2, 4, 1, 3, 5]  # Orden para agrupar por compañía
    ax.legend([handles[idx] for idx in order], [labels[idx] for idx in order], loc='upper left')

    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"\nHa ocurrido un error inesperado: {e}")
    print("Asegúrate de tener las librerías 'torch', 'chronos-forecasting' y 'accelerate' instaladas.")

In [ ]:
# Leer el dataset
import pandas as pd

df = pd.read_csv("dataset_completo_google_amazon.csv")

# Mostrar las primeras filas del dataset
df.head()
df

In [ ]:
# Graficar el dataset
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 7))
df['GOOGL_Close'].plot(ax=ax);
df['AMZN_Close'].plot(ax=ax);
ax.set_xlabel("anos")
ax.set_ylabel("Precios de cierre");

In [ ]:
import torch
from chronos import BaseChronosPipeline

# Crear pipeline: carga el modelo y lo mueve al dispositivo (GPU o CPU) y
# define el tipo de dato para almacenar variables numéricas

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",  # el nombre del modelo
    device_map="cpu",  # CPU
    torch_dtype=torch.bfloat16,
)

In [ ]:
# Predicción básica
# Tomar todo el dataset como entrada exceptuando el último año
N = 365
contexto = torch.tensor(df["GOOGL_Close"][:-N].values, dtype=torch.float)

# Generar la predicción con "predict"
pred1 = pipeline.predict(
    context=contexto, # La serie de entrada
    prediction_length=N, # Número de instantes de tiempo a predecir
)
print(pred1.shape)

In [ ]:
for i in range(20):
    plt.plot(pred1[0, i], color="gray", alpha=0.3)

In [ ]:
# Calcular predicción promedio
pred1_mean = pred1.mean(dim=1)
print(pred1_mean.shape)

In [ ]:
# Graficar dataset original y predicción
plt.figure(figsize=(12, 5))

# Original series
plt.plot(df["GOOGL_Close"].values, label="Serie original")

# Predicción
start_idx = len(contexto)
forecast_idx = range(start_idx, start_idx + N)
plt.plot(forecast_idx, pred1_mean.squeeze(), label="Predicción promedio", color="red", marker="o", alpha=0.5)

plt.xlabel("Mes")
plt.ylabel("Nro. pasajeros")
plt.legend()
plt.grid(True)
plt.tight_layout();

In [ ]:
# Veamos una predicción a 3 años

# Contexto
N = 36
contexto = torch.tensor(df["GOOGL_Close"][:-N].values, dtype=torch.float)

# Predicción
pred1 = pipeline.predict(
    context=contexto,
    prediction_length=N,
)

# Calcular predicción promedio
pred1_mean = pred1.mean(dim=1)

# Gráfico serie original y predicción
plt.figure(figsize=(12, 5))

plt.plot(df["GOOGL_Close"].values, label="Serie original")

start_idx = len(contexto)
forecast_idx = range(start_idx, start_idx + N)
plt.plot(forecast_idx, pred1_mean.squeeze(), label="Predicción promedio", color="red", marker="o", alpha=0.5)

plt.xlabel("Mes")
plt.ylabel("Nro. pasajeros")
plt.legend()
plt.grid(True)
plt.tight_layout();

In [ ]:
# calcular predicción con base en la mediana y comparar
pred1_median = pred1.median(dim=1).values

# Gráfico original y predicción
plt.figure(figsize=(12, 5))

# Serie original
plt.plot(df["GOOGL_Close"].values, label="Serie original")

# Pronóstico
start_idx = len(contexto)
forecast_idx = range(start_idx, start_idx + N)
plt.plot(forecast_idx, pred1_mean.squeeze(), label="Pronóstico promedio", color="red", marker="o", alpha=0.5)
plt.plot(forecast_idx, pred1_median.squeeze(), label="Pronóstico mediana", color="green", marker="o", alpha=0.5)

plt.xlabel("Mes")
plt.ylabel("Nro. pasajeros")
plt.legend()
plt.grid(True)
plt.tight_layout();

In [ ]:
cuantiles, mean = pipeline.predict_quantiles(
    context=contexto,
    prediction_length=N,
    quantile_levels=[0.1, 0.5, 0.9], # Calcular cuantiles 10%, 50% (mediana) y 90%
)

In [ ]:
# Y graficar
import matplotlib.pyplot as plt

low, median, high = cuantiles[0, :, 0], cuantiles[0, :, 1], cuantiles[0, :, 2]

plt.figure(figsize=(12, 5))
start_idx = len(contexto)
forecast_idx = range(start_idx, start_idx + N)
plt.plot(contexto, color="royalblue", label="Serie original")
plt.plot(forecast_idx, median, color="tomato", label="Pronóstico mediana")
plt.fill_between(forecast_idx, low, high, color="tomato", alpha=0.3, label="Intervalo de predicción 80%")
plt.legend()
plt.grid();

In [ ]:
# Crear un DataFrame con los pronósticos
forecast_data = {
    'Bajo (10%)': low,
    'Mediana (50%)': median,
    'Alto (90%)': high
}

# Generar las fechas para el índice del pronóstico
forecast_dates = pd.bdate_range(start=df.index[start_idx], periods=N, freq='B')

forecast_table = pd.DataFrame(forecast_data, index=forecast_dates)

# Mostrar la tabla de pronósticos
print("\n--- Tabla de Pronósticos (Primeros 10 días) ---")
display(forecast_table.head(10))

print("\n--- Tabla de Pronósticos (Últimos 10 días) ---")
display(forecast_table.tail(10))

In [ ]:
# --- Generar Pronóstico para 2025 (usando todos los datos históricos) ---
print("Iniciando el pronóstico para 2025 (usando todos los datos históricos)...")

# El contexto es la serie histórica completa de Google
full_context_googl = torch.tensor(google_series.values, dtype=torch.bfloat16)

# Definir los días hábiles de 2025 para la predicción
dias_habiles_2025 = pd.bdate_range(start='2025-01-01', end='2025-12-31')
prediction_length_2025 = len(dias_habiles_2025)

print(f"Realizando pronóstico para los {prediction_length_2025} días hábiles de 2025...")

# Predecir los cuantiles para 2025
quantiles_googl_2025_tensor, _ = pipeline.predict_quantiles(
    context=full_context_googl.to(device), # Use the 'device' variable
    prediction_length=prediction_length_2025,
    quantile_levels=[0.1, 0.5, 0.9],
)

quantiles_googl_2025 = quantiles_googl_2025_tensor.cpu().numpy()
print("Pronóstico para 2025 completado.")

# --- Procesar y Mostrar los Resultados de 2025 en una Tabla ---
low_2025, median_2025, high_2025 = quantiles_googl_2025[0, :, 0], quantiles_googl_2025[0, :, 1], quantiles_googl_2025[0, :, 2]

forecast_df_2025 = pd.DataFrame(
    data={'Bajo (10%)': low_2025, 'Mediana (50%)': median_2025, 'Alta (90%)': high_2025},
    index=dias_habiles_2025
)

print("\n--- Tabla de Pronósticos para GOOGL en 2025 (Primeros 10 días) ---")
display(forecast_df_2025.head(10))

print("\n--- Tabla de Pronósticos para GOOGL en 2025 (Últimos 10 días) ---")
display(forecast_df_2025.tail(10))